In [298]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_theme(palette='pastel')

pd.set_option('display.max_rows', 50)
pd.set_option('display.max_columns', 50)

In [299]:
# 한글 폰트 출력
import matplotlib.font_manager as fm

# 설치된 폰트 출력
font_list = [font.name for font in fm.fontManager.ttflist]
font_list

plt.rcParams['font.family'] = 'Pretendard'

In [300]:
# 데이터 불러오기
merchant_df = pd.read_csv('../data/raw/big_data_set1_f.csv', encoding='cp949')
sale_df = pd.read_csv('../data/raw/big_data_set2_f.csv', encoding='cp949')
cust_df = pd.read_csv('../data/raw/big_data_set3_f.csv', encoding='cp949')

In [302]:
# merchant_df 컬럼명 변경
merchant_df.columns = ['mct_id', 'mct_address', 'mct_name', 'brand_code', 'sigungu', 'industry_name', 'commercial_area', 'open_date', 'close_date']

In [304]:
# sale_df 컬럼명 변경
sale_df.columns = ['mct_id', 'trans_date', 'operation_band', 'sale_amt_band', 'sale_cnt_band', 'unique_customer_band', 'aov_band', 'cancel_rate_band', 'delivery_sale_amount_ratio', 'industry_sale_amt_ratio', 'industry_sale_cnt_ratio',
                   'industry_sale_rank_pct', 'area_sale_rank_pct', 'industry_close_pct', 'area_close_pct']

In [306]:
# cust_df 컬럼명 변경
cust_df.columns = ['mct_id', 'trans_date', 'male_u20', 'male_u30', 'male_u40', 'male_u50', 'male_u60', 'female_u20', 'female_u30', 'female_u40', 'female_u50', 'female_u60', 
                   'visit_re', 'visit_new', 'route_resident', 'route_worker', 'route_floating']

# Feature Engineering

### merchant_df 기준으로 DataFrame 결합

#### sale_df 전처리

In [334]:
## 구간 데이터 전처리
# 복사본 생성
sale_df_prc = sale_df.copy()

# 구간 데이터 앞에 숫자만 빼오기
sale_df_prc.loc[:, 'operation_band':'cancel_rate_band'] = sale_df_prc.loc[:, 'operation_band':'cancel_rate_band'].apply(lambda x: x.str.split('_', expand=True)[0])

# 결측치 최빈값으로 대체 후 구간 데이터들의 자료형 int형으로 변환
# 가정: 결측값의 경우 취소율이 적다고 판단!!
band_cols = sale_df_prc.loc[:, 'operation_band':'cancel_rate_band'].columns
sale_df_prc[band_cols] = sale_df_prc[band_cols].fillna(1).astype(int)

## 비율 데이터 전처리
# 배달매출금액 비율의 -999999.9는 배달매출 미존재 의미 => 0으로 대체 가능 (배달매출금액 비율이 처음부터 0인 데이터도 있긴 함)
sale_df_prc['delivery_sale_amount_ratio'] = sale_df_prc['delivery_sale_amount_ratio'].map(lambda x: 0 if x < 0 else x)

# 동일 상권 내 해지 가맹점 비중의 -999999.9는 상권 미존재 의미 => 100으로 할 수 도 있겠으나 일단 0으로 대체
sale_df_prc['area_close_pct'] = sale_df_prc['area_close_pct'].map(lambda x: 0 if x < 0 else x)

# 가맹점별 매출정보 평균계산
sale_df_prc = sale_df_prc.drop('trans_date', axis=1).groupby('mct_id').mean().reset_index()

# sale_df_prc 데이터와 merchant_df 결합한 데이터 merged_df에 저장
merged_df = merchant_df.merge(sale_df_prc, on='mct_id')

### cust_df 전처리

In [335]:
## 고객 데이터 전처리
# 복사본 생성
cust_df_prc = cust_df.copy()

# 결측치는 0으로 대체
cust_cols = cust_df_prc.iloc[:, 2:].columns
cust_df_prc[cust_cols] = cust_df_prc[cust_cols].clip(lower=0)

#  가맹점별 고객정보 평균계산
cust_df_prc = cust_df_prc.drop('trans_date', axis=1).groupby('mct_id').mean().reset_index()

# cust_df_prc 데이터 merged_df에 추가
merged_df = merged_df.merge(cust_df_prc, on='mct_id')

### 범주형 데이터 그룹화
- 업종 통합 시 sale_df의 업종관련 정보와의 차이 발생 가능

In [336]:
## 업종 그룹화 
meat = ['한식-육류/고기',  '꼬치구이']
cafe = ['카페',  '주스',  '차',  '테마카페',  '커피전문점', '테이크아웃커피',  '구내식당/푸드코트']
k_food = ['백반/가정식',  '기사식당', '한식-두부요리', '한식-단품요리일반',  '한정식',    '한식-죽',  '한식-국수/만두',  '한식-국밥/설렁탕',  '한식-찌개/전골',  '한식-냉면',  '한식뷔페',  '한식-감자탕',   '한식-해물/생선']
w_food = ['양식',  '스테이크', '치킨',  '햄버거',  '피자']
j_food = ['일식당',  '일식-우동/소바/라면',  '일식-초밥/롤',  '일식-덮밥/돈가스',  '일식-샤브샤브',  '일식-참치회']
c_food = ['중식당',  '중식-딤섬/중식만두',  '중식-훠궈/마라탕']
drink = ['호프/맥주',  '요리주점',  '민속주점',  '포장마차',  '이자카야',  '와인바', '주류',  '와인샵']
product= ['농산물',  '청과물',  '수산물',  '건어물',  '축산물']
enter = ['일반 유흥주점',  '룸살롱/단란주점']
convenience = ['샌드위치/토스트',  '도시락', '분식']
world_food = ['동남아/인도음식',  '기타세계요리']
dessert = ['도너츠',  '탕후루',  '와플/크로플',  '마카롱',  '아이스크림/빙수',  '떡/한과',  '떡/한과 제조',  '베이커리']
others = ['식품 제조',  '반찬',  '미곡상',  '유제품',  '인삼제품', '건강식품',  '건강원', '담배',  '식료품']

# replace 진행
groups_to_replace = [(meat, 'meat'), (cafe, 'cafe'), (k_food, 'k_food'), (w_food, 'w_food'), (j_food, 'j_food'),
                                       (c_food, 'c_food'), (drink, 'drink'), (product, 'product'), (enter, 'enter'), (convenience, 'convenience'),
                                       (world_food, 'world_food'), (dessert, 'dessert'), (others, 'others')]
replacement = {}
for ind, cat in groups_to_replace:
    for i in ind:
        replacement[i] = cat

merged_df['industry_name'].replace(replacement, inplace=True)

In [337]:
# 상권 데이터 1개만 존재하는 경우 주변 상권과 통합 (장한평자동차와 답십리는 모두 구석진 곳인데 가까우므로 결합)
areas_to_replace = {'화양시장': '성수', '자양': '성수', '서면역': '성수', '미아사거리': '성수',
                                    '방배역': '뚝섬', '건대입구': '뚝섬', '풍산지구': '뚝섬', '오남': '한양대',
                                    '동대문역사문화공원역': '금남시장',  '압구정로데오': '금남시장',  '장한평자동차': '답십리'}
merged_df['commercial_area'].replace(areas_to_replace, inplace=True)

# 상권 결측치는 Unknown으로 대체
merged_df['commercial_area'].fillna('Unknown', inplace=True)

In [338]:
# merchant_df.loc[merchant_df['mct_address'].str.contains('장터길')]

### is_closed 컬럼 생성

In [339]:
# 폐업여부 나타내는 is_closed 컬럼 추가
merged_df['is_closed'] = merged_df['close_date'].notna().astype(int)

### 불필요 컬럼 제거

In [340]:
# 기존 merchant_df에 존재했던 컬럼 중 모델에 사용할 수 없는 컬럼 제거
merged_df.drop(['mct_id', 'mct_address', 'mct_name', 'brand_code', 'sigungu', 'open_date', 'close_date'], axis=1, inplace=True)

In [341]:
# 상관계수 절댓값이 0.7 이상인 컬럼들이 있으면 둘 중 하나 제거
merged_df.drop(['sale_amt_band', 'unique_customer_band', 'industry_sale_cnt_ratio', 'industry_sale_rank_pct'], axis=1, inplace=True)

In [342]:
# plt.figure(figsize=(15, 10))
# test = merged_df.copy()
# test.drop(['sale_amt_band', 'unique_customer_band', 'industry_sale_cnt_ratio', 'industry_sale_rank_pct'], axis=1, inplace=True)
# sns.heatmap(test.select_dtypes(exclude='object').corr(), annot=True, fmt='.1f', cmap='RdBu')
# plt.show()

### One-Hot Encoding

In [343]:
# One-Hot-Encoding
merged_df = pd.get_dummies(merged_df, drop_first=True, dtype=int)